# CFNA install, corpus build, training, and inference

This notebook sets up the CFNA project in a Jupyter environment, builds the allowed public-domain corpus, trains a checkpoint, and runs a short inference/chat test.

**Corpus policy:** the build uses only the sources registered in `cfna.corpus.sources.safe_commercial_sources()` and the corpus builder enforces allowed source hosts plus commercial-safe public-domain/CC0 license IDs. No blogs, social media, forums, scraped web, or copyrighted sources are used.


In [ ]:
# 1) Install the project dependencies in the active Jupyter kernel.
# If you already installed them, this cell is safe to skip.
%pip install -r ../requirements.txt
%pip install -e .. --no-build-isolation


In [ ]:
# 2) Confirm the notebook is running from the repo and imports work.
from pathlib import Path
import sys

REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import torch
import cfna

print("repo:", REPO)
print("cfna:", cfna.__version__)
print("numpy:", np.__version__)
print("torch:", torch.__version__)


In [ ]:
# 3) Inspect the only sources this notebook will use.
from cfna.corpus.sources import safe_commercial_sources, EXCLUDED_KINDS

sources = safe_commercial_sources()
for source in sources:
    print(f"{source.source_id}: {source.url} | {source.license_id} | bucket={source.bucket}")

print("\nExplicitly excluded source kinds:", ", ".join(EXCLUDED_KINDS))


In [ ]:
# 4) Build the allowed public-domain corpus.
# This downloads curated NLTK-hosted Project Gutenberg and US-government speech archives
# via raw.githubusercontent.com, then writes corpus/manifest.jsonl.
import subprocess

subprocess.run([sys.executable, str(REPO / "scripts" / "build_corpus.py"), "--out", str(REPO / "corpus")], check=True)


In [ ]:
# 5) Verify every manifest record is allowed before training.
import json
from collections import Counter
from urllib.parse import urlparse

manifest_path = REPO / "corpus" / "manifest.jsonl"
records = [json.loads(line) for line in manifest_path.read_text(encoding="utf-8").splitlines() if line.strip()]
allowed_hosts = {"raw.githubusercontent.com"}
allowed_license_ids = {"public-domain-us", "public-domain-usgov", "CC0-1.0"}

assert records, "corpus manifest is empty"
for record in records:
    assert record["bucket"] == "safe_commercial", record["document_id"]
    assert record["commercial_use"] is True, record["document_id"]
    assert record["attribution_required"] is False, record["document_id"]
    assert record["license_id"] in allowed_license_ids, record["document_id"]
    assert urlparse(record["source_locator"]).hostname in allowed_hosts, record["document_id"]

print(f"documents: {len(records)}")
print("splits:", dict(Counter(r["split"] for r in records)))
print("licenses:", dict(Counter(r["license_id"] for r in records)))
print("bytes:", sum(r["n_bytes"] for r in records))


In [ ]:
# 6) Train a checkpoint.
# For a quick smoke test, use 0.25-1 minute. For a better demo, use 20+ minutes.
TRAIN_MINUTES = 1.0
CKPT = REPO / "checkpoints" / "cfna_chat.pt"

subprocess.run([
    sys.executable, str(REPO / "scripts" / "train_checkpoint.py"),
    "--corpus", str(REPO / "corpus"),
    "--minutes", str(TRAIN_MINUTES),
    "--out", str(CKPT),
], check=True)


In [ ]:
# 7) Run the built-in chat/inference demo.
subprocess.run([
    sys.executable, str(REPO / "scripts" / "chat_demo.py"),
    "--ckpt", str(CKPT),
    "--temp", "0.7",
    "--max-new", "120",
    "--seed", "0",
], check=True)


In [ ]:
# 8) Ask your own prompt from inside the notebook.
from cfna.chat import Conversation, load_checkpoint

model, ckpt = load_checkpoint(CKPT)
conversation = Conversation(model, system="A conversation.", temperature=0.7, max_new=120)

prompt = "Hello CFNA. Can you describe liberty in one paragraph?"
reply = conversation.say(prompt)
print("User:", prompt)
print("Assistant:", reply)
